# Yargıtay Kararları — Embedding & Qdrant

Bu notebook, Google Drive'a yüklediğin `Nlp veriler.zip` dosyasını alır,
Yargıtay kararlarını **chunk'lara böler**, multilingual-e5-base ile embed eder ve
**`kira_yargitay`** koleksiyonu olarak Qdrant'a yükler.

Mevcut `kira_hukuku` koleksiyonuna (TBK maddeleri) dokunulmaz. Streamlit app (`app.py`)
iki koleksiyonu paralel sorgulayıp hem madde hem içtihat atıflı cevap üretecek.

**Klasör yapısı beklentisi:**
```
Nlp veriler/
├── TBK_344_kira_artis/
│   ├── 2014_4518_2015_27_12.01.2015.txt
│   └── ...
├── TBK_347_on_yil_kiraci/
│   └── ...
└── TBK_315_ihtarname_tahliye/
    └── ...
```

**Dosya adı paterni:** `<esas_yıl>_<esas_no>_<karar_yıl>_<karar_no>_<gg.aa.yyyy>.txt`

## Hücre 1 — Kütüphaneleri Kur

In [ ]:
!pip install qdrant-client sentence-transformers -q
print('✓ Kurulum tamamlandı')

## Hücre 2 — Konfigürasyon

Sol menü → 🔑 **Secrets** kısmına `QDRANT_API_KEY` ekli olmalı (önceki notebook'tan zaten ekli).

In [ ]:
# Colab bağımlılıklarını (userdata) tamamen sildik.

# URL'miz artık kendi bilgisayarımız (localhost)
QDRANT_URL = "http://localhost:6333"

# Lokal Qdrant'ta şifreye (API Key) ihtiyacımız yok, o satırı da sildik.
KARAR_KOLEKSIYON = "kira_yargitay"

print('✓ Konfigürasyon yüklendi')
print(f'  Qdrant URL  : {QDRANT_URL}')
print(f'  Koleksiyon  : {KARAR_KOLEKSIYON}')

## Hücre 4 — Dosyaları Oku ve Meta Çıkar

Dosya adından `Esas No`, `Karar No`, `Tarih` çıkarılır. Klasör adından TBK madde no eşlenir.

In [ ]:
import re
import json # JSON kaydetmek için eklendi
from collections import Counter
from pathlib import Path # EKSİK 1: Dosya yolu kütüphanesi eklendi

# EKSİK 2: Aramanın yapılacağı ana dizin tanımlandı
# "./" komutu, python dosyasının bulunduğu klasörü ve alt klasörlerini arar
veri_dizini = Path("./") 

KATEGORI_MAP = {
    'TBK_344_kira_artis'        : {'madde': '344', 'konu': 'Kira bedelinin belirlenmesi / artışı'},
    'TBK_347_on_yil_kiraci'     : {'madde': '347', 'konu': 'On yıllık uzama süresi sonunda tahliye'},
    'TBK_315_ihtarname_tahliye' : {'madde': '315', 'konu': 'Temerrüt nedeniyle fesih / ihtarname'},
}

def dosya_adindan_meta(dosya_adi: str):
    stem = Path(dosya_adi).stem
    pattern = r'(\d{4})[:_/\-](\d+)[_/\-](\d{4})[:_/\-](\d+)[_/\-]?(\d{1,2}\.\d{1,2}\.\d{4})'
    m = re.search(pattern, stem)
    if not m:
        return None
    return {
        'esas_no'  : f'{m.group(1)}/{m.group(2)}',
        'karar_no' : f'{m.group(3)}/{m.group(4)}',
        'tarih'    : m.group(5),
    }

kararlar = []
hata = 0

print("Dosyalar taranıyor, lütfen bekleyin...\n")

for klasor_adi, bilgi in KATEGORI_MAP.items():
    # __MACOSX klasörünü FİLTRELE
    bulunanlar = [
        p for p in veri_dizini.rglob(klasor_adi)
        if p.is_dir() and '__MACOSX' not in p.parts
    ]
    if not bulunanlar:
        print(f'⚠ Klasör bulunamadı: {klasor_adi}')
        continue
    klasor = bulunanlar[0]

    for txt_dosya in sorted(klasor.glob('*.txt')):
        # macOS AppleDouble metadata dosyalarını atla
        if txt_dosya.name.startswith('._'):
            continue

        meta = dosya_adindan_meta(txt_dosya.name)
        if meta is None:
            print(f'  ⚠ Meta çıkarılamadı: {txt_dosya.name}')
            hata += 1
            continue

        tam_metin = txt_dosya.read_text(encoding='utf-8').strip()
        if not tam_metin:
            continue

        # Embedding kalitesi için gereksiz satır atlamalarını temizle
        tam_metin = re.sub(r'\s+', ' ', tam_metin).strip()

        kararlar.append({
            **meta,
            'ilgili_madde'   : bilgi['madde'],
            'konu'           : bilgi['konu'],
            'kategori'       : klasor_adi,
            'dosya_adi'      : txt_dosya.name,
            'baslik'         : f"Yargıtay Kararı - E: {meta['esas_no']} K: {meta['karar_no']}",
            'icerik'         : tam_metin,
            'veri_turu'      : 'yargitay',
            'karakter_sayisi': len(tam_metin),
        })

print(f'\n✓ {len(kararlar)} karar yüklendi (Hata: {hata})')
print('\nKategori dağılımı:')
for kat, n in Counter(k['kategori'] for k in kararlar).items():
    print(f'  {kat}: {n}')

if kararlar:
    ornek = kararlar[0]
    print(f'\nÖrnek karar Önizleme:')
    print(f"  E. {ornek['esas_no']}  |  K. {ornek['karar_no']}  |  {ornek['tarih']}")
    print(f"  Konu: {ornek['konu']}  |  Uzunluk: {ornek['karakter_sayisi']} kr")

    # Qdrant'a yükleyebilmek için tertemiz bir JSON dosyası oluşturalım
    json_yolu = "yargitay_kararlari.json"
    with open(json_yolu, "w", encoding="utf-8") as f:
        json.dump(kararlar, f, ensure_ascii=False, indent=4)
    print(f"\n🚀 Tüm kararlar Qdrant'a yüklenmeye hazır şekilde '{json_yolu}' dosyasına kaydedildi!")

## Hücre 5 — Chunking

Yargıtay kararları uzun olabildiği için (bazıları 5+ sayfa), e5'in 512 token sınırına takılmamak adına ~1500 karaktere bölünür. **Cümle sınırları korunur** — chunk ortasında cümle kesilmez. Her parça, ana kararın `esas_no`/`karar_no`/`tarih` bilgisini taşır, böylece LLM her parçadan tam atıf yapabilir.

In [ ]:
import json
import re

# 1. Bir önceki adımda oluşturduğumuz ham JSON dosyasını okuyoruz
with open("yargitay_kararlari.json", "r", encoding="utf-8") as f:
    kararlar = json.load(f)

# 2. Akıllı Parçalama (Chunking) Fonksiyonu
def chunkla(metin: str, max_karakter: int = 1500):
    paragraflar = [p.strip() for p in metin.split('\n\n') if p.strip()]
    parcalar, mevcut = [], ''

    for para in paragraflar:
        if len(para) > max_karakter:
            cumleler = re.split(r'(?<=[.!?])\s+', para)
            for c in cumleler:
                if len(mevcut) + len(c) + 1 <= max_karakter:
                    mevcut += (' ' if mevcut else '') + c
                else:
                    if mevcut: parcalar.append(mevcut.strip())
                    mevcut = c
        else:
            if len(mevcut) + len(para) + 2 <= max_karakter:
                mevcut += ('\n\n' if mevcut else '') + para
            else:
                if mevcut: parcalar.append(mevcut.strip())
                mevcut = para

    if mevcut.strip():
        parcalar.append(mevcut.strip())
    return parcalar

# 3. Parçalama İşlemini Başlat
tum_parcalar = []
for karar in kararlar:
    # HATA BURADAYDI: 'tam_metin' yerine 'icerik' yazıyoruz
    parcalar = chunkla(karar['icerik']) 
    
    for i, p in enumerate(parcalar):
        tum_parcalar.append({
            'esas_no'      : karar['esas_no'],
            'karar_no'     : karar['karar_no'],
            'tarih'        : karar.get('tarih', 'Bilinmiyor'),
            'ilgili_madde' : karar.get('ilgili_madde', ''),
            'konu'         : karar.get('konu', ''),
            'kategori'     : karar.get('kategori', ''),
            'dosya_adi'    : karar.get('dosya_adi', ''),
            'baslik'       : karar.get('baslik', ''),
            'parca_no'     : i + 1,
            'toplam_parca' : len(parcalar),
            'icerik'       : p, # Artık sadece bu küçük parçayı içerik yapıyoruz
            'veri_turu'    : 'yargitay',
        })

print(f"✓ {len(kararlar)} adet uzun karar işlendi.")
print(f"✓ Toplam {len(tum_parcalar)} adet küçük anlamsal parça (chunk) elde edildi.")
print(f"  Ortalama : {len(tum_parcalar)/len(kararlar):.1f} parça/karar")
print(f"  En uzun  : {max(len(p['icerik']) for p in tum_parcalar)} karakter")
print(f"  En kısa  : {min(len(p['icerik']) for p in tum_parcalar)} karakter")

# 4. Parçalanmış (Chunked) veriyi yeni bir JSON olarak kaydet
yeni_json_yolu = "yargitay_kararlari_chunked.json"
with open(yeni_json_yolu, "w", encoding="utf-8") as f:
    json.dump(tum_parcalar, f, ensure_ascii=False, indent=4)

print(f"\n🚀 Parçalanmış veriler Qdrant embedding'i için '{yeni_json_yolu}' dosyasına kaydedildi!")

## Hücre 6 — Embedding Modelini Yükle

**Aynı model**: `multilingual-e5-base` (768 boyut). TBK koleksiyonuyla aynı modeli kullanmak şart — aksi halde vektör uzayları farklı olur, hibrit arama çalışmaz.

In [ ]:
from sentence_transformers import SentenceTransformer

print('Model yükleniyor (ilk seferde ~1-2 dk)...')
model = SentenceTransformer('intfloat/multilingual-e5-base')
print(f'✓ Yüklendi — boyut: {model.get_sentence_embedding_dimension()}')

## Hücre 7 — Yargıtay Koleksiyonunu Oluştur

`kira_hukuku`'ya **dokunulmuyor**. Yeni `kira_yargitay` koleksiyonu açılıyor.
İleride sadece m.347 kararlarını filtrelemek istersek diye payload index'leri de kuruluyor.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PayloadSchemaType

# 1. Değişkenleri tekrar tanımlıyoruz (Lokal bağlantı)
QDRANT_URL = "http://localhost:6333"
KARAR_KOLEKSIYON = "kira_yargitay"

# 2. API Key olmadan sadece URL ile bağlanıyoruz
client = QdrantClient(url=QDRANT_URL)

mevcut = [c.name for c in client.get_collections().collections]
print(f'Mevcut koleksiyonlar: {mevcut}')

if KARAR_KOLEKSIYON in mevcut:
    print(f'⚠ "{KARAR_KOLEKSIYON}" zaten var — silip yeniden oluşturuluyor')
    client.delete_collection(KARAR_KOLEKSIYON)

# 3. Koleksiyonu E5 boyutuna göre oluştur
client.create_collection(
    collection_name=KARAR_KOLEKSIYON,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

# 4. Aramayı hızlandıran Index'leri ekle
for alan in ('ilgili_madde', 'kategori', 'veri_turu'):
    client.create_payload_index(
        collection_name=KARAR_KOLEKSIYON,
        field_name=alan,
        field_schema=PayloadSchemaType.KEYWORD,
    )

print(f'✓ "{KARAR_KOLEKSIYON}" oluşturuldu (index\'lerle birlikte)')

## Hücre 8 — Embedding Üret ve Qdrant'a Yükle

GPU varsa hızlanır (Colab → Runtime → Change runtime type → T4 GPU).

In [ ]:
import json
import time
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from sentence_transformers import SentenceTransformer

# 1. Bağlantı Ayarları
QDRANT_URL = "http://localhost:6333"
KARAR_KOLEKSIYON = "kira_yargitay"
client = QdrantClient(url=QDRANT_URL)

# 2. Modeli Yükle (Mac Hızlandırması ile)
print('Model yükleniyor...')
model = SentenceTransformer('intfloat/multilingual-e5-base', device="mps")
print('✓ Model yüklendi\n')

# 3. Chunklanmış JSON'ı Oku
with open("yargitay_kararlari_chunked.json", "r", encoding="utf-8") as f:
    tum_parcalar = json.load(f)

print(f'{len(tum_parcalar)} parça için embedding üretiliyor...')

# E5 Kuralı: Metinlerin başına 'passage: ' ekliyoruz. 
# AI'ın konuyu unutmaması için başlığı da içeriğe dahil ettim.
metinler = [f"passage: {p['baslik']} (Parça {p['parca_no']}/{p['toplam_parca']}) - {p['icerik']}" for p in tum_parcalar]

# 4. Vektörleştirme (Embedding)
t0 = time.time()
vektorler = model.encode(
    metinler,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f'✓ Embedding tamamlandı ({time.time()-t0:.1f} sn)')

# 5. Qdrant'a Yükleme (Senin Batch Mantığınla)
print("\nQdrant'a yükleniyor (100'lük batch'ler halinde)...")
BATCH = 100
for i in range(0, len(tum_parcalar), BATCH):
    grup_p = tum_parcalar[i:i+BATCH]
    grup_v = vektorler[i:i+BATCH]
    points = [
        PointStruct(id=i+j, vector=v.tolist(), payload=p)
        for j, (p, v) in enumerate(zip(grup_p, grup_v))
    ]
    client.upsert(collection_name=KARAR_KOLEKSIYON, points=points)
    print(f'  {i+len(points)}/{len(tum_parcalar)} parça yüklendi')

# 6. Kontrol
toplam = client.count(collection_name=KARAR_KOLEKSIYON, exact=True).count
print(f'\n🚀 MÜKEMMEL! Qdrant\'ta toplam {toplam} Yargıtay parçası hazır!')

## Hücre 9 — Sağlık Kontrolü: Sadece Yargıtay'da Arama

Koleksiyonun düzgün çalıştığını doğrulayalım.

In [ ]:
def yargitay_ara(soru: str, top_k: int = 5):
    vektor = model.encode(
        f'query: {soru}',
        normalize_embeddings=True,
    ).tolist()

    sonuclar = client.query_points(
        collection_name=KARAR_KOLEKSIYON,
        query=vektor,
        limit=top_k,
        with_payload=True,
    ).points

    print(f'🔍 "{soru}"')
    print('─' * 70)
    for s in sonuclar:
        p = s.payload
        print(f"  E.{p['esas_no']} / K.{p['karar_no']} / {p['tarih']}  "
              f"(m.{p['ilgili_madde']}, skor: {s.score:.3f})")
        print(f"    {p['icerik'][:160]}...")
        print()

yargitay_ara('10 yıllık kiracı tahliyesi')
yargitay_ara('kira artış oranı TÜFE üzeri')
yargitay_ara('temerrüt ihtarnamesi 30 gün süre')

## Hücre 10 — Hibrit Kontrol: TBK + Yargıtay Birlikte

App.py'nin yaptığı şeyin aynısı: aynı soruyu **iki koleksiyona** birden sorup sonucu yan yana görelim.

In [ ]:
KANUN_KOLEKSIYON = "kira_hukuku"

def hibrit_arama(soru: str, top_kanun: int = 3, top_karar: int = 4):
    vektor = model.encode(
        f'query: {soru}',
        normalize_embeddings=True,
    ).tolist()

    kanun = client.query_points(
        collection_name=KANUN_KOLEKSIYON,
        query=vektor, limit=top_kanun, with_payload=True,
    ).points
    karar = client.query_points(
        collection_name=KARAR_KOLEKSIYON,
        query=vektor, limit=top_karar, with_payload=True,
    ).points

    print(f'🔍 "{soru}"')
    print('═' * 70)
    print(f'📖 TBK MADDELERİ ({len(kanun)})')
    for s in kanun:
        p = s.payload
        print(f"   m.{p['madde_no']} — {p['baslik']}  (skor: {s.score:.3f})")
    print(f'\n⚖️  YARGITAY KARARLARI ({len(karar)})')
    for s in karar:
        p = s.payload
        print(f"   E.{p['esas_no']} / K.{p['karar_no']} — m.{p['ilgili_madde']}  (skor: {s.score:.3f})")
    print()

hibrit_arama('Ev sahibi kirayı sınırsız artırabilir mi?')
hibrit_arama('10 yıl dolduğunda kiracıyı sebepsiz çıkarabilirim')
hibrit_arama('Kira ödenmedi, kaç günlük süre verilmeli ihtarnamede?')